# アノテーション

アノテーションでは、11_record_cameraで撮影した走行データにアノテーションを実施し、転移学習をおこないます。

収集した走行データを用いて、アノテーションをし、データセットを作成します。

### ボードの識別

In [ ]:
import os

# ---------- 1. Jetson.GPIO 読み取り ----------
try:
    import Jetson.GPIO as GPIO
    BOARD_NAME = GPIO.gpio_pin_data.get_data()[0]
except Exception as e:
    # 失敗したら Orin Nano と決め打ち
    print(f"[WARN] Jetson モデル判定エラー: {e} → 強制的に JETSON_ORIN_NANO として続行")
    os.environ["JETSON_MODEL_NAME"] = "JETSON_ORIN_NANO"
    import Jetson.GPIO as GPIO          # もう一度ロード
    BOARD_NAME = "JETSON_ORIN_NANO"     # 確定

# ---------- 2. ボード別定義 ----------
mode_descriptions = {
    "JETSON_NX":       ["15W_2CORE", "15W_4CORE", "15W_6CORE", "10W_2CORE", "10W_4CORE"],
    "JETSON_XAVIER":   ["MAXN", "MODE_10W", "MODE_15W", "MODE_30W"],
    "JETSON_NANO":     ["MAXN", "5W"],
    "JETSON_ORIN":     ["MAXN", "MODE_15W", "MODE_30W", "MODE_40W"],
    "JETSON_ORIN_NANO":["MODE_15W", "MODE_25W", "MODE_MAX"]
}

product_names = {
    "JETSON_NX":        "Jetson Xavier NX",
    "JETSON_XAVIER":    "Jetson AGX Xavier",
    "JETSON_NANO":      "Jetson Nano",
    "JETSON_ORIN":      "Jetson AGX Orin",
    "JETSON_ORIN_NANO": "Jetson Orin Nano"
}

# (I2C バス番号, 初期 Power モードインデックス)
board_settings = {
    "JETSON_NX":        (8, 3),
    "JETSON_XAVIER":    (8, 2),
    "JETSON_NANO":      (1, 0),
    "JETSON_ORIN":      (7, 0),
    "JETSON_ORIN_NANO": (7, 2)
}

# ---------- 3. パラメータ取得 ----------
i2c_busnum, power_mode = board_settings.get(BOARD_NAME, (None, None))
mode_list       = mode_descriptions.get(BOARD_NAME, [])
product_name    = product_names.get(BOARD_NAME, "未知のボード")

# ---------- 4. 出力 ----------
if i2c_busnum is not None and 0 <= power_mode < len(mode_list):
    mode_str = mode_list[power_mode]
    print("------------------------------------------------------------")
    print(f"{product_name} を認識: I2C バス番号 = {i2c_busnum}, "
          f"Power モード = {mode_str} ({power_mode})")
    print("------------------------------------------------------------")
else:
    raise RuntimeError(f"未対応の Jetson モデル、または Power モード定義不足: {BOARD_NAME}")

In [ ]:
!echo "jetson" | sudo -S nvpmodel -m $power_mode

In [ ]:
!echo "jetson" | sudo -S nvpmodel -q

In [ ]:
!echo "jetson" | sudo -S jetson_clocks

走行データはcameraフォルダに録画されています。今度は、cameraフォルダのデータにアノテーションをおこないます。

In [ ]:
import os
import matplotlib.pyplot as plt
import matplotlib.image as mpimg
import matplotlib.patches as patches
import re
import ipywidgets
from utils import preprocess
from fabo.annotation import draw_grids
from ipywidgets import Button, Layout, Textarea, HBox, VBox, Label, GridBox
from jupyter_clickable_image_widget import ClickableImageWidget
from jetcam.utils import bgr8_to_jpeg
import cv2
import torchvision.transforms as transforms
from xy_dataset import XYDataset
import time
import threading
import torch
import torchvision

In [ ]:
IMG_WIDTH = 224
IMG_HEIGHT = 224

SLEEP = [50,100,200,300,400,500]
SKIP = [1,2,3,4,5,10,15,20,30,50,100]

LOAD_CATEGORIES = ['xy','speed']
SAVE_CATEGORIES = ['xy','speed']

LOAD_DATASETS = ['X','Y','Z']
SAVE_DATASETS = []

LOAD_TASK = ['camera','dataset','interactive']
SAVE_TASK = ['dataset']

check_flag = False
running = False
sleep_time = 30
current_path = os.getcwd()

def get_dirs(path):
    files = os.listdir(path)
    dirs = [f for f in files if os.path.isdir(os.path.join(path, f))]
    dirs = [f for f in files if f != ".ipynb_checkpoints"]
    dirs = sorted(dirs)
    
    return dirs

try:  
    path = os.path.join(current_path,"dataset")
    dirs = get_dirs(path)
    if not dirs:
        SAVE_DATASETS = ['dataset1']
    else:
        SAVE_DATASETS = dirs
        
except:
    SAVE_DATASETS = ['dataset1']

TRANSFORMS = transforms.Compose([
    transforms.ColorJitter(0.2, 0.2, 0.2, 0.2),
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
])

datasets = {}
for name in SAVE_DATASETS:
    for task in SAVE_TASK:
        datasets[name] = XYDataset(task + '/' + name, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
        
dataset = datasets[SAVE_DATASETS[0]]

In [ ]:
l = Layout(flex='0 1 auto', height='100px', min_height='100px', width='auto')
process_widget = ipywidgets.Textarea(description='ログ', value='', layout=l)

process_no = 0

MAX_LOG_LINES = 200  # 保持する最大行数

def write_log(msg):
    global process_widget, process_no
    process_no += 1
    new_line = f"{process_no}: {msg}"
    
    # 既存ログを分割して最新200行だけ保持
    lines = process_widget.value.split("\n")
    lines.insert(0, new_line)
    if len(lines) > MAX_LOG_LINES:
        lines = lines[:MAX_LOG_LINES]
    process_widget.value = "\n".join(lines)


In [ ]:
sleep_dropdown = ipywidgets.Dropdown(options=SLEEP, description='sleep(ms)', index=1)
skip_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)
skip_movie_dropdown = ipywidgets.Dropdown(options=SKIP, description='skip(枚)', index=1)

picture_widget = ClickableImageWidget(width=224, height=224)
picture_widget.format = "jpeg"

no_widget = ipywidgets.IntText(description='no')
x_widget = ipywidgets.IntText(description='data x')
y_widget = ipywidgets.IntText(description='data y')
speed_widget = ipywidgets.IntText(description='data speed')
ai_x_widget = ipywidgets.IntText(description='AI　x')
ai_y_widget = ipywidgets.IntText(description='AI　y')
ai_speed_widget = ipywidgets.IntText(description='AI speed')
model_widget = ipywidgets.Text(description='model')
model_widget.value = "model.pth"
load_model_button = ipywidgets.Button(description='load model')
speed_slider = ipywidgets.IntSlider(description='speed', min=0, max=224, step=1, value=0, orientation='vertical')
add_speed_button = ipywidgets.Button(description='速度追加')

In [ ]:
from packaging import version

torchvision_version = version.parse(torchvision.__version__)

device = torch.device('cuda')
output_dim = 2*len(LOAD_CATEGORIES)  # LOAD_CATEGORIESは事前に定義されている必要があります

if torchvision_version >= version.parse("0.13"):
    from torchvision.models.resnet import ResNet18_Weights, resnet18

    default_weights = torchvision.models.ResNet18_Weights.DEFAULT
    model = torchvision.models.resnet18(weights=default_weights)
    model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
else:
    model = torchvision.models.resnet18(pretrained=True)
    model.fc = torch.nn.Linear(512, output_dim)

model = model.to(device)

In [ ]:
def load_model(c):
    global torchvision_version, load_model_widget, model, device, output_dim
    
    model_name = load_model_widget.value
    device = torch.device('cuda')
    
    if torchvision_version >= version.parse("0.13"):
        # torchvision 0.13以降の場合
        from torchvision.models.resnet import ResNet18_Weights, resnet18
        
        if model_name == "[new]":
            # 新しい重みを使ってモデルをロード
            default_weights = torchvision.models.ResNet18_Weights.DEFAULT
            model = torchvision.models.resnet18(weights=default_weights)
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            model = model.to(device)
            write_log("[new]が選択されたのでresnet18の最新の重みから始めます(torchvision 0.13以降)。")
        else:
            model = torchvision.models.resnet18(weights=None)  # pretrained=Falseの代わり
            model.fc = torch.nn.Linear(model.fc.in_features, output_dim)
            model = model.to(device)
            state_dict = torch.load(model_name, weights_only=True)
            model.load_state_dict(state_dict)
            write_log(model_name + "のモデルを読込ました(torchvision 0.13以降)。")

    else:
        # torchvision 0.13より前の場合
        if model_name == "[new]":
            model = torchvision.models.resnet18(pretrained=True)
            model.fc = torch.nn.Linear(512, output_dim)
            model = model.to(device)
            write_log("[new]が選択されたのでresnet18のpretrainedから始めます。")
        else:
            model = torchvision.models.resnet18(pretrained=False)
            model.fc = torch.nn.Linear(512, output_dim)
            model = model.to(device)
            model.load_state_dict(torch.load(model_name))
            write_log(model_name + "のモデルを読込ました。")
    
    get_jetson_nano_memory_usage()

def save_model(c):
    global save_model_name_widget, model, device
    path = "./model/"
    if not os.path.exists(path):
        subprocess.call(['mkdir', '-p', path])
    torch.save(model.state_dict(), path + save_model_name_widget.value)
    write_log(path + save_model_name_widget.value + "に保存しました。")

In [ ]:
import time

BATCH_SIZE = 8

epochs_widget = ipywidgets.IntText(description='epochs', value=1)
eval_button = ipywidgets.Button(description='evaluate')
train_button = ipywidgets.Button(description='train')
loss_widget = ipywidgets.FloatText(description='loss')
progress_widget = ipywidgets.FloatProgress(min=0.0, max=1.0, description='progress')

best_loss = float('inf')  

def train_eval(is_training):
    global BATCH_SIZE, model, dataset, optimizer, best_loss, current_path
    
    optimizer = torch.optim.Adam(model.parameters())
    
    xy_path = os.path.join(current_path, save_task_widget.value, save_datasets_widget.value, "xy")
    speed_path = os.path.join(current_path, save_task_widget.value, save_datasets_widget.value, "speed")

    xy_is_dir = os.path.isdir(xy_path)
    speed_is_dir = os.path.isdir(speed_path)
    
    xy_file_count = 0
    speed_file_count = 0
    
    if xy_is_dir:
        xy_file_count = sum(os.path.isfile(os.path.join(xy_path, name)) for name in os.listdir(xy_path))
    if speed_is_dir:
        speed_file_count = sum(os.path.isfile(os.path.join(speed_path, name)) for name in os.listdir(speed_path))
    
    write_log("-------------------------")
    write_log("学習を開始します。")
    write_log("データセット: " + save_task_widget.value + '/' + save_datasets_widget.value)
    write_log("XYデータ数: " + str(xy_file_count) + " Speedデータ数: " + str(speed_file_count))
    write_log("-------------------------")

    # 総学習時間を計測開始
    total_start_time = time.time()

    dataset = XYDataset(save_task_widget.value + '/' + save_datasets_widget.value,
                        SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
    train_button.disabled = True
    eval_button.disabled = True
        
    # 有効データ確認
    valid_indices = []
    for i in range(len(dataset)):
        try:
            item = dataset[i]
            if item is not None and item[0] is not None:
                valid_indices.append(i)
        except Exception:
            pass
        
    # サブセット化
    valid_dataset = torch.utils.data.Subset(dataset, valid_indices)
    
    try:
        train_loader = torch.utils.data.DataLoader(
            valid_dataset,
            batch_size=BATCH_SIZE,
            shuffle=True
        )
        time.sleep(1)

        if is_training:
            model = model.train()
        else:
            model = model.eval()

        epoch_count = 0
        
        while epochs_widget.value > 0:
            epoch_start_time = time.time()
            epoch_count += 1
            i = 0
            sum_loss = 0.0

            for images, category_idx, xy in iter(train_loader):
                if images is None or xy is None:
                    print("Warning: None type data found at index", i)
                    continue

                images = images.to(device)
                xy = xy.to(device)

                if is_training:
                    optimizer.zero_grad()

                outputs = model(images)

                loss = 0.0
                for batch_idx, cat_idx in enumerate(list(category_idx.flatten())):
                    loss += torch.mean(
                        (outputs[batch_idx][2 * cat_idx:2 * cat_idx + 2] - xy[batch_idx]) ** 2
                    )
                loss /= len(category_idx)

                if is_training:
                    loss.backward()
                    optimizer.step()

                count = len(category_idx.flatten())
                i += count
                sum_loss += float(loss)
                progress_widget.value = i / len(dataset)
                loss_widget.value = sum_loss / i
            
            # エポック終了時ログ
            epoch_end_time = time.time()
            epoch_duration = epoch_end_time - epoch_start_time
            write_log(f"{epoch_count} Epoch目: {epoch_duration:.2f}秒")

            # ベストモデル保存
            if is_training and loss_widget.value < best_loss:
                best_loss = loss_widget.value
                model_dir = './model'
                if not os.path.exists(model_dir):
                    os.makedirs(model_dir)
                torch.save(model.state_dict(), model_dir + "/" + 'best_model.pth')
                write_log(f"新しいベストモデルが保存されました。Epoch loss: {best_loss:.4f}")

            if is_training:
                epochs_widget.value -= 1
            else:
                break

    except Exception as e:
        write_log(f"学習中にエラーが発生: {e}")
    finally:
        model = model.eval()
        train_button.disabled = False
        eval_button.disabled = False

        # ✅ 学習全体の時間を出力
        total_duration = time.time() - total_start_time
        h = int(total_duration // 3600)
        m = int((total_duration % 3600) // 60)
        s = int(total_duration % 60)
        write_log("-------------------------")
        write_log(f"総学習時間: {h}時間 {m}分 {s}秒（{total_duration:.2f} 秒）")
        write_log("-------------------------")


train_button.on_click(lambda c: train_eval(is_training=True))
eval_button.on_click(lambda c: train_eval(is_training=False))
    
train_eval_widget = ipywidgets.VBox([
    epochs_widget,
    progress_widget,
    loss_widget,
    ipywidgets.HBox([train_button, eval_button])
])

display(train_eval_widget)

01_find_pwmを実行して、pwmの値を設定してください。

In [ ]:
import Fabo_PCA9685
import time
import smbus
import time
import json

with open('pwm_params.json') as f:
    json_str = json.load(f)

    stop = json_str["pwm_speed"]["stop"]
    left = json_str["pwm_steering"]["left"]
    center = json_str["pwm_steering"]["center"]
    right = json_str["pwm_steering"]["right"]

if stop == 0:
    INITIAL_VALUE=400
else:
    INITIAL_VALUE=stop

In [ ]:
def map_rc(x, in_min, in_max, out_min, out_max):
    return (x - in_min) * (out_max - out_min) // (in_max - in_min) + out_min

def handle(x):
    x = map_rc(x, 224, 0, right, left)

In [ ]:
from os.path import join
import subprocess
import datetime
import glob
import functools
import ipywidgets as widgets
from ipywidgets import VBox, Layout

def extract_numbers(filename):
    matches = re.findall(r'(\d+)', filename)
    if matches and len(matches) >= 3: 
        return int(matches[-1])  
    else:
        return float('inf') 

def get_file_names(path):
    file_names = os.listdir(path)
    file_names = [os.path.join(path, file_name) for file_name in file_names]
    image_names = []

    image_names = sorted(file_names, key=lambda f: extract_numbers(os.path.basename(f)))
    image_names = [f for f in image_names if os.path.splitext(f)[1].lower() == ".jpg"]
    
    return image_names

@functools.lru_cache(maxsize=1024)
def load_img_from_disk(path):
    return cv2.imread(path, cv2.IMREAD_COLOR) 

def load_img(no):
    global running, img, load_flag, xy_filenames, play_num, current_path
    load_task_value = load_task_widget.value
    category_value = load_category_widget.value
    datasets_value = load_datasets_widget.value
    
    xy_path = os.path.join(current_path, load_task_value, datasets_value, "xy")
    speed_path = os.path.join(current_path, load_task_value, datasets_value, "speed")
    
    t_all = time.perf_counter()
    
    xy_imagenames = get_file_names(xy_path)

    # --- 未定義参照の防止: デフォルトを先に用意 ---
    x = 0
    y = 0
    speed = 0
    # ----------------------------------------------

    if no >= len(xy_imagenames):
        no_widget.value = no - 1
        write_log("ファイルが存在しません。" + str(len(xy_imagenames)-1) + "以内の値を設定してください。")
        running = False
        return
        
    xy_name = xy_imagenames[no]

    # ここは basename から安全に抽出（拡張子やUUIDがあってもOK）
    base = os.path.basename(xy_name)
    m = re.match(r'^(\d+)_(\d+)_', base)
    if m:
        try:
            x = int(m.group(1))
            y = int(m.group(2))
            x_widget.value = x
            y_widget.value = y
        except Exception:
            # 何かおかしくても0に戻す
            x = 0
            y = 0
            x_widget.value = 0
            y_widget.value = 0
    else:
        # 取得できない場合は 0 を使う
        x_widget.value = 0
        y_widget.value = 0

    # speed 側も同様に basename から第2要素を拾う（存在しない環境は無視）
    try:
        speed_imagenames = get_file_names(speed_path)
        if no < len(speed_imagenames):
            sp_base = os.path.basename(speed_imagenames[no])
            sm = re.match(r'^\d+_(\d+)_', sp_base)
            if sm:
                speed = int(sm.group(1))
                speed_widget.value = speed
                speed_slider.value = speed
            else:
                speed_widget.value = 0
        else:
            speed_widget.value = 0
    except Exception:
        speed_widget.value = 0

    # --- ここから先は元の処理（読み込み・推論・描画など） ---
    t0 = time.perf_counter()
    img = load_img_from_disk(xy_name)
    load_ms = (time.perf_counter() - t0) * 1000
    
    if img is None:
        write_log("Image could not be loaded: " + xy_name)
        return

    marked_img = img.copy()
    
    black_color = (0, 0, 0)
    blue_color = (255, 0, 0)
    green_color = (0, 255, 0)

    if int(x) != 0 or int(y) != 0:
        marked_img = cv2.circle(marked_img, (int(x), int(y)), 8, green_color, 3)
    
    try:
        t0 = time.perf_counter()
        preprocessed = preprocess(img)
        output = model(preprocessed).detach().cpu().numpy().flatten()
        infer_ms = (time.perf_counter() - t0) * 1000
        
        t0 = time.perf_counter()
        result_x = output[0]
        result_y = output[1]
        result_speed = output[3]
        result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
        result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
        result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
        
        handle(result_x)
        ai_x_widget.value = result_x
        ai_y_widget.value = result_y
        ai_speed_widget.value = result_speed
        
        marked_img = cv2.circle(marked_img, (int(result_x), int(result_y)), 8, blue_color, 3)
        marked_img = draw_grids(marked_img)
        
        # Speed
        if result_speed> 224:
            result_speed = 224
        elif result_speed < 0:
            result_speed = 0
            
        marked_img = cv2.line(marked_img,(218,0),(218,224),black_color,5)
        marked_img = cv2.line(marked_img,(219,224-result_speed),(219,224),blue_color,3)
        marked_img = cv2.putText(marked_img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))
    
        if int(speed) != 0:
            marked_img = cv2.line(marked_img,(1,0),(1,224),black_color,5)
            marked_img = cv2.line(marked_img,(2,224-int(speed)),(2,224),green_color,3)        
    except Exception as e:
        write_log(f"エラーが発生しました: {e}")
    
    draw_ms = (time.perf_counter() - t0) * 1000
    
    t0 = time.perf_counter()
    picture_widget.value = bgr8_to_jpeg(marked_img)
    enc_ms = (time.perf_counter() - t0) * 1000
    
    total_ms = (time.perf_counter() - t_all) * 1000
    write_log(
        f"{no} 枚目 {os.path.basename(xy_name)} "
        f"load:{load_ms:.1f}ms infer:{infer_ms:.1f}ms "
        f"draw:{draw_ms:.1f}ms enc:{enc_ms:.1f}ms total:{total_ms:.1f}ms"
    )

    if running == True:
        play_num += 1
        if play_num % 10 == 0:
            write_log(f"{play_num} 回目の再生")
    else:
        next_image_button.disabled = False
        prev_image_button.disabled = False
        write_log(str(no) + "枚目の" + xy_name + "を読込ました。") 

def _parse_name_parts(path):
    """
    期待する形式: "<num>_<num>_<uuid>.jpg" 例) 113_87_c420e068-aa5d-11f0-ac25-8f40a87d2421.jpg
    戻り値: (first:int|None, second:int|None, uuid:str|None)
    """
    base = os.path.basename(path)
    m = re.match(r'^(\d+)_(\d+)_([0-9a-fA-F-]+)\.(jpg|jpeg|png)$', base)
    if not m:
        return None, None, None
    return int(m.group(1)), int(m.group(2)), m.group(3)

def del_pic(c):
    """
    xy 側の UUID と完全一致する speed 側ファイルを 1 件だけ削除（インデックス非依存・誤爆防止）。
    """
    global no_widget, current_path

    no = int(no_widget.value)
    load_task_value = load_task_widget.value
    datasets_value  = load_datasets_widget.value
    xy_path    = os.path.join(current_path, load_task_value, datasets_value, "xy")
    speed_path = os.path.join(current_path, load_task_value, datasets_value, "speed")

    # xy リスト（命名規則順でソート）
    xy_names = get_file_names(xy_path)
    if not xy_names:
        write_log("削除対象の xy 画像がありません。")
        return
    if not (0 <= no < len(xy_names)):
        write_log(f"削除位置が不正です。0〜{len(xy_names)-1} を指定してください。")
        return

    xy_name = xy_names[no]
    _, _, xy_uuid = _parse_name_parts(xy_name)
    if not xy_uuid:
        write_log(f"xy ファイル名の形式が想定外で UUID を取得できません: {xy_name}")
        return

    # まず xy を削除
    try:
        os.remove(xy_name)
        write_log(f"[xy] {xy_name} を削除しました。")
    except Exception as e:
        write_log(f"[xy] {xy_name} の削除に失敗: {e}")
        return

    # speed 側：UUID 完全一致を 1 件だけ探して削除
    deleted_speed = False
    if os.path.isdir(speed_path):
        try:
            for f in os.listdir(speed_path):
                if not f.lower().endswith((".jpg", ".jpeg", ".png")):
                    continue
                sp_full = os.path.join(speed_path, f)
                _, _, sp_uuid = _parse_name_parts(sp_full)
                if sp_uuid == xy_uuid:
                    try:
                        os.remove(sp_full)
                        write_log(f"[speed] {sp_full} を削除しました。")
                        deleted_speed = True
                    except Exception as e:
                        write_log(f"[speed] {sp_full} の削除に失敗: {e}")
                    break  # ※ 1件のみ
            if not deleted_speed:
                write_log(f"[speed] UUID={xy_uuid} に一致するファイルは見つかりませんでした。")
        except Exception as e:
            write_log(f"[speed] 検索中にエラー: {e}")
    else:
        write_log("[speed] フォルダが存在しないためスキップしました。")

    # 件数表示と UI 更新
    file_count()
    new_xy_names = get_file_names(xy_path)
    if not new_xy_names:
        no_widget.value = 0
        canvas.clear()
        blank = np.ones((224, 224, 3), dtype=np.uint8) * 255
        canvas.put_image_data(cv2.cvtColor(blank, cv2.COLOR_BGR2RGB))
        write_log("xy の画像がすべて削除されました。")
        return

    if no >= len(new_xy_names):
        no = len(new_xy_names) - 1
    no_widget.value = no

    try:
        load_img(no)
    except Exception as e:
        write_log(f"表示更新に失敗: {e}")


    
def load_dataset(c):
    global img,load_flag,current_path
    dataset_path = os.path.join(current_path,load_task_widget.value,load_datasets_widget.value)
    write_log("データセット: " + dataset_path + "を読込みます。")
    load_flag = True
    no = 0
    write_log("初回の読込みには時間がかかります。(30秒〜1分)")
    load_img(no)
    get_jetson_nano_memory_usage()

def next_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) + skip_dropdown.value
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)
    
def before_pic(c):
    global no,x,y,load_flag,skip,check_flag
    load_flag = True
    check_flag = False
    no = no_widget.value
    no = int(no) - skip_dropdown.value
    if no < 0:
        no = 0
    no_widget.value = no
    next_image_button.disabled = True
    prev_image_button.disabled = True
    load_img(no)

def file_count():
    global current_path
    
    try:
        xy_path = os.path.join(current_path,save_task_widget.value,save_datasets_widget.value,"xy")
        speed_path = os.path.join(current_path,save_task_widget.value,save_datasets_widget.value,"speed")
        
        xy_is_dir = os.path.isdir(xy_path)
        
        if xy_is_dir:
            xy_file_count = sum(os.path.isfile(os.path.join(xy_path,name)) for name in os.listdir(xy_path))
            datasets_xy_count_widget.value = xy_file_count
        else:
            datasets_xy_count_widget.value = 0
        
        speed_is_dir = os.path.isdir(speed_path)
        
        if speed_is_dir:
            speed_file_count = sum(os.path.isfile(os.path.join(speed_path,name)) for name in os.listdir(speed_path))
            datasets_speed_count_widget.value = speed_file_count
        else:
            datasets_speed_count_widget.value = 0
            
    except Exception as e:
        #print("An error occurred:", e)
        datasets_xy_count_widget.value = 0
        datasets_speed_count_widget.value = 0
    
def save_snapshot(_, content, msg):
    global img,x,y,load_flag,save_datasets_widget,save_task_widget,save_category_widget
    if content['event'] == 'click' and load_flag == True:
        load_flag = False
        data = content['eventData']
        x = min(max(data['offsetX'], 0), IMG_WIDTH)  # 0 <= x <= IMG_WIDTH
        y = data['offsetY']

        remarked_img = img.copy()
        remarked_img = cv2.circle(remarked_img, (int(x), int(y)), 8, (0, 255, 0), 3)
        picture_widget.value = bgr8_to_jpeg(remarked_img)
        name = save_datasets_widget.value
        if save_task_widget.value == "":
            write_out("データセット名を指定してください")
        else:
            write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[0] + "カテゴリにデータを追加しました。")
            dataset = datasets[name]
            dataset.save_entry("xy", img, x, y)
            #write_log("新しい座標で保存しました。")
            file_count()
        
def save_speed(c):
    global img,speed_slider,save_datasets_widget,save_task_widget
    speed = speed_slider.value
    remarked_img = img.copy()
    remarked_img = _put_dataset_badge(remarked_img, save_task_widget.value, save_datasets_widget.value)
    picture_widget.value = bgr8_to_jpeg(remarked_img)

    
    name = save_datasets_widget.value
    dataset = datasets[name]
    dataset.save_entry("speed", img, 0, speed)
    write_log("["+save_task_widget.value + "/" + name + "]の" + SAVE_CATEGORIES[1] + "カテゴリにデータを追加しました。")
    
    file_count()
        
def live():
    global no,running, skip, sleep_time, play_num
    load_flag = True
    play_num = 0
    no = no_widget.value
    while running:
        no += skip
        no_widget.value = no
        try:
            load_img(no)
        except:
            write_log("no: " + no + "のファイルの読込に失敗")
        time.sleep(sleep_time/1000)  
    
def play(c):
    global running, execute_thread, skip, sleep_time, check_flag
    skip = skip_dropdown.value
    sleep_time = sleep_dropdown.value
    running = True
    check_flag = False
    execute_thread = threading.Thread(target=live)
    execute_thread.start()
    
def stop(c):
    global running, execute_thread, load_flag, check_flag
    running = False
    load_flag = True
    check_flag = False
    try:
        execute_thread.join()
        write_log("STOP")
    except:
        write_log("現在再生されていません。")

def create_dataset(c):
    global datasets_name_widget, save_datasets_widget
    new_dataset_name = datasets_name_widget.value
    
    if new_dataset_name not in SAVE_DATASETS:
        SAVE_DATASETS.append(new_dataset_name)

        save_datasets_widget.options = SAVE_DATASETS
        save_datasets_widget.value = new_dataset_name
    
    datasets = {}
    for name in SAVE_DATASETS:
        for task in SAVE_TASK:
            datasets[name] = XYDataset(task + '/' + name, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
            
    dataset = datasets[new_dataset_name]  # 新しいデータセットを指定
    write_log("Datasetを作成しました：" + new_dataset_name)
    
# 降順で定義
SPEED_PRESETS = [224, 180, 120, 100, 80, 60, 50, 40, 30, 0]

speed_preset_buttons = [Button(description=f"TH{v}") for v in SPEED_PRESETS]

# ボタンサイズ
for b in speed_preset_buttons:
    b.layout = Layout(width="100px", height="32px")

def _make_speed_handler(val):
    def _h(_):
        global img, datasets, save_datasets_widget, save_task_widget, picture_widget
        if 'img' not in globals() or img is None:
            write_log("画像が未読込のため速度を追加できません。")
            return
        if not save_task_widget.value:
            write_log("保存先 task が未指定です。")
            return

        name = save_datasets_widget.value
        task = save_task_widget.value

        if name not in datasets:
            datasets[name] = XYDataset(
                task + '/' + name,
                SAVE_CATEGORIES, TRANSFORMS, random_hflip=True
            )

        try:
            # 保存（speed は (x=0, y=val) として保存）
            datasets[name].save_entry("speed", img, 0, int(val))
            write_log(f"[{task}/{name}] の speed に {val} を追加しました。")

            # ★ プレビューに保存先のバッジを常時表示
            marked = img.copy()
            # （任意）UIで選んだプリセット値も視覚化したい場合はバーを描画
            try:
                disp = max(0, min(224, int(val)))
                marked = cv2.line(marked, (1,0), (1,224), (0,0,0), 5)           # 軸
                marked = cv2.line(marked, (2,224-disp), (2,224), (0,255,0), 3)  # 現在値
                marked = cv2.putText(marked, f"speed:{disp}", (8,215),
                                      cv2.FONT_HERSHEY_SIMPLEX, 0.4, (255,255,255), 1, cv2.LINE_AA)
            except Exception:
                pass

            # バッジ重ねる
            marked = _put_dataset_badge(marked, task, name)
            picture_widget.value = bgr8_to_jpeg(marked)

            # オートスキップ（遅延あり）
            if 'auto_advance_toggle' in globals() and auto_advance_toggle.value == 'skip':
                _delayed_next_pic()
        except Exception as e:
            write_log(f"速度{val} の追加に失敗: {e}")
    return _h


# ハンドラ登録
for b, v in zip(speed_preset_buttons, SPEED_PRESETS):
    b.on_click(_make_speed_handler(v))


speed_presets_box = GridBox(
    children=speed_preset_buttons,
    layout=Layout(
        grid_template_columns="repeat(2, auto)",  # 2列にする
        grid_gap="1px 1px",                       # 行間・列間の余白
        justify_content="flex-start",             # 左寄せ
        align_items="flex-start"
    )
)

# XY(0,97)を追加するボタン
add_xy0122_button = ipywidgets.Button(description='XY(0,97)追加')
# XY(224,97)を追加するボタン
add_xy224122_button = ipywidgets.Button(description='XY(224,97)追加')

def add_xy0122(c):
    global img, datasets, save_datasets_widget, save_task_widget
    if 'img' not in globals() or img is None:
        write_log("画像が未読込のため追加できません。")
        return
    if not save_task_widget.value:
        write_log("保存先 task が未指定です。")
        return

    name = save_datasets_widget.value
    if name not in datasets:
        datasets[name] = XYDataset(save_task_widget.value + '/' + name, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)

    try:
        # データ保存
        datasets[name].save_entry("xy", img, 0, 97)
        write_log(f"[{save_task_widget.value}/{name}] の xy に (0,97) を追加しました。")

        # 表示用にマーク追加
        marked_img = img.copy()
        marked_img = cv2.circle(marked_img, (0, 97), 8, (0, 255, 0), 3)
        marked_img = _put_dataset_badge(marked_img, task, name)
        
        # ClickableImageWidgetを使う場合
        picture_widget.value = bgr8_to_jpeg(marked_img)

        file_count()
        
        # 遅延スキップ
        if 'auto_advance_toggle' in globals() and auto_advance_toggle.value == 'skip':
            _delayed_next_pic()

    except Exception as e:
        write_log(f"(0,97) 追加に失敗: {e}")

def add_xy224122(c):
    global img, datasets, save_datasets_widget, save_task_widget
    if 'img' not in globals() or img is None:
        write_log("画像が未読込のため追加できません。")
        return
    if not save_task_widget.value:
        write_log("保存先 task が未指定です。")
        return

    name = save_datasets_widget.value
    if name not in datasets:
        datasets[name] = XYDataset(save_task_widget.value + '/' + name, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)

    try:
        # データ保存
        datasets[name].save_entry("xy", img, 224, 97)
        write_log(f"[{save_task_widget.value}/{name}] の xy に (224,97) を追加しました。")

        # 表示用にマーク追加
        marked_img = img.copy()
        marked_img = cv2.circle(marked_img, (224, 97), 8, (0, 255, 0), 3)
        marked_img = _put_dataset_badge(marked_img, task, name)

        # ClickableImageWidgetを使う場合
        picture_widget.value = bgr8_to_jpeg(marked_img)

        file_count()
        
        # 遅延スキップ
        if 'auto_advance_toggle' in globals() and auto_advance_toggle.value == 'skip':
            _delayed_next_pic()
        
    except Exception as e:
        write_log(f"(244,97) 追加に失敗: {e}")

        
# ボタンをイベント登録
add_xy0122_button.on_click(add_xy0122)
add_xy224122_button.on_click(add_xy224122)

def _delayed_next_pic():
    import threading, time
    def _go():
        try:
            time.sleep(CLICK_HOLD_MS / 1000.0)
            next_pic(None)
        except Exception as e:
            write_log(f"遅延スキップ中にエラー: {e}")
    threading.Thread(target=_go, daemon=True).start()
    
def on_click_picture(_, content, msg):
    """
    ClickableImageWidget用: 画像がアスペクト維持でfit表示されている前提で、
    余白(padding)を補正して実ピクセル座標へマッピング。
    クリック後の動作は auto_advance_toggle で切り替え。
    """
    global img, picture_widget, datasets, save_datasets_widget, save_task_widget

    if content.get('event') != 'click':
        return
    if 'img' not in globals() or img is None:
        write_log("画像未読込です。")
        return

    # 表示上の座標（ウィジェット内）
    disp_x = float(content['eventData']['offsetX'])
    disp_y = float(content['eventData']['offsetY'])

    # ウィジェットの表示サイズ（なければ既定の画像サイズ）
    widget_w = float(getattr(picture_widget, 'width', 224))
    widget_h = float(getattr(picture_widget, 'height', 224))

    # 画像の実サイズ
    img_h, img_w = img.shape[:2]

    # アスペクト維持fitのスケールと余白を計算
    scale = min(widget_w / img_w, widget_h / img_h)
    content_w = img_w * scale
    content_h = img_h * scale
    pad_x = (widget_w - content_w) / 2.0
    pad_y = (widget_h - content_h) / 2.0

    # 余白を引いたコンテンツ内座標
    content_x = disp_x - pad_x
    content_y = disp_y - pad_y

    # コンテンツ外クリックは無視（必要に応じて）
    if content_x < 0 or content_y < 0 or content_x > content_w or content_y > content_h:
        write_log("画像領域外のクリックです。")
        return

    # 実ピクセル座標へ変換
    x = int(round(content_x / scale))
    y = int(round(content_y / scale))
    x = max(0, min(img_w - 1, x))
    y = max(0, min(img_h - 1, y))

    # 保存
    name = save_datasets_widget.value
    if name not in datasets:
        datasets[name] = XYDataset(save_task_widget.value + '/' + name, SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
    datasets[name].save_entry("xy", img, x, y)
    write_log(f"[{save_task_widget.value}/{name}] xy に ({x},{y}) を追加")

    # （任意）可視化したい場合は描画
    marked = img.copy()
    marked = cv2.circle(marked, (x, y), 8, (0,255,0), 3)
    
     # 保存先データセットのバッジを重ねる
    marked = _put_dataset_badge(marked, save_task_widget.value, save_datasets_widget.value)

    picture_widget.value = bgr8_to_jpeg(marked)

    file_count()
    
    # 遅延スキップ
    if auto_advance_toggle.value == 'skip':
        import threading, time
        def _delayed_next():
            try:
                time.sleep(CLICK_HOLD_MS / 1000.0)
                next_pic(None)
            except Exception as e:
                write_log(f"遅延スキップ中にエラー: {e}")
        threading.Thread(target=_delayed_next, daemon=True).start()


# クリック後確認表示
def _put_dataset_badge(img, task: str, name: str):
    """
    画像左上に 'task/dataset' の半透明バッジを重ねる
    """
    badge_text = f"{task}/{name}" if task else name
    overlay = img.copy()
    font = cv2.FONT_HERSHEY_SIMPLEX
    scale = 0.5
    thickness = 1
    (tw, th), baseline = cv2.getTextSize(badge_text, font, scale, thickness)

    x0, y0 = 4, 4
    pad_x, pad_y = 6, 6
    x1, y1 = x0 + tw + pad_x*2, y0 + th + pad_y*2

    # 半透明の黒い背景 + 白枠
    cv2.rectangle(overlay, (x0, y0), (x1, y1), (0, 0, 0), -1)
    cv2.rectangle(overlay, (x0, y0), (x1, y1), (255, 255, 255), 1)
    cv2.putText(overlay, badge_text, (x0 + pad_x, y0 + th + pad_y - 1), font, scale, (255, 255, 255), thickness, cv2.LINE_AA)

    # 透過合成
    alpha = 0.65
    cv2.addWeighted(overlay, alpha, img, 1 - alpha, 0, img)
    return img



# ハンドラの登録    
#picture_widget.on_msg(save_snapshot)
picture_widget.on_msg(on_click_picture)

# 画像の操作
play_button = ipywidgets.Button(description='▶')
stop_button = ipywidgets.Button(description='⏹')
next_image_button = ipywidgets.Button(description='>')
prev_image_button = ipywidgets.Button(description='<')
load_image_button = ipywidgets.Button(description='読込')
update_image_button = ipywidgets.Button(description='更新')
delete_image_button = ipywidgets.Button(description='削除')

save_model_button = ipywidgets.Button(description='save model')

load_model_widget = ipywidgets.Dropdown(options=[],description='読込モデル')
load_model_time_widget = ipywidgets.Text(description='作成日時')
save_model_name_widget = ipywidgets.Text(description='保存モデル名',value="model.pth")

dataset_create_button = ipywidgets.Button(description='Create dataset')
datasets_name_widget = ipywidgets.Text(description='Name')

dataset_create_button.on_click(create_dataset)

play_button.on_click(play)
stop_button.on_click(stop)

add_speed_button.on_click(save_speed)

load_model_button.on_click(load_model)
save_model_button.on_click(save_model)
load_image_button.on_click(load_dataset)
next_image_button.on_click(next_pic)
prev_image_button.on_click(before_pic)
delete_image_button.on_click(del_pic)

load_datasets_widget = ipywidgets.Dropdown(options=LOAD_DATASETS, description='dataset', index=0)
save_datasets_widget = ipywidgets.Dropdown(options=SAVE_DATASETS, description='dataset')
datasets_xy_count_widget = ipywidgets.IntText(description='XYデータ数')
datasets_speed_count_widget = ipywidgets.IntText(description='速度データ数')

# クリック後の動作を選ぶUI
# 値: 'stop' = 進まない, 'skip' = skip_dropdownの枚数だけ次へ
auto_advance_toggle = widgets.ToggleButtons(
    options=[('クリック後に止まる', 'stop'), ('クリック後にスキップ', 'skip')],
    value='skip',
    description='クリック後動作',
)

# 待ち時間スライダ
click_hold_ms_slider = widgets.IntSlider(
    description='クリック後の待ち時間(ms)',
    min=0, max=1000, step=10, value=20
)
CLICK_HOLD_MS = click_hold_ms_slider.value

def _change_hold_ms(change):
    global CLICK_HOLD_MS
    CLICK_HOLD_MS = change['new']
click_hold_ms_slider.observe(_change_hold_ms, names='value')


def set_dataset(change):
    global dataset
    datasets[change['new']] = XYDataset(save_task_widget.value + '/' + change['new'], SAVE_CATEGORIES, TRANSFORMS, random_hflip=True)
    #dataset = datasets[change['new']]
    #write_log(change['new'])
save_datasets_widget.observe(set_dataset, names='value')

load_task_widget = ipywidgets.Dropdown(options=LOAD_TASK, description='task')
save_task_widget = ipywidgets.Dropdown(options=SAVE_TASK,  value=SAVE_TASK[0], description='task')

def change_load_task(change):
    global dataset, current_path, load_task_widget, load_datasets_widget
    try:
        path = os.path.join(current_path,load_task_widget.value)
        dirs = get_dirs(path)
        load_datasets_widget.options = dirs
    except:
        write_log(path + "が存在していません。")
        load_datasets_widget.options = []
load_task_widget.observe(change_load_task, names='value')
change_load_task(LOAD_TASK[0])

def change_save_task(change):
    global dataset, current_path
    try:
        path = os.path.join(current_path,save_task_widget.value)
        if not os.path.exists(path):
            subprocess.call(['mkdir', '-p', path])
        dirs = get_dirs(path)
        save_datasets_widget.options = dirs
    except:
        write_log(path + "が存在していません。")
        save_datasets_widget.options = ['']
save_task_widget.observe(change_save_task, names='value')
change_save_task(SAVE_TASK[0])

def change_save_dataset(change):
    global dataset, current_path
    file_count()
save_datasets_widget.observe(change_save_dataset, names='value')
change_save_dataset(SAVE_DATASETS[0])

def change_sleep(change):
    global sleep_time, sleep_dropdown
    sleep_time = sleep_dropdown.value
sleep_dropdown.observe(change_sleep, names='value')

def change_skip(change):
    global sleep, sleep_dropdown
    skip = skip_dropdown.value
skip_dropdown.observe(change_skip, names='value')

def model_list(change):
    global load_model_widget
    try:
        files = glob.glob('./model/*.pth', recursive=True)
        files.insert(0,"[new]")
        load_model_widget.options = files
        load_model_time_widget.value = ""
    except:
        load_model_widget.options = []
model_list("list")

def change_file(change):
    global load_model_widget
    try:
        file = load_model_widget.value
        ts = os.path.getctime(file)
        d = datetime.datetime.fromtimestamp(ts)
        s = d.strftime('%Y-%m-%d %H:%M:%S')
        load_model_time_widget.value = s
    except:
        load_model_time_widget.value = ""
load_model_widget.observe(change_file, names='value')

def update_image(change):
    global load_flag, no
    load_flag = True
    no = no_widget.value
    load_img(no)       
update_image_button.on_click(update_image)

load_category_widget = ipywidgets.Dropdown(options=LOAD_CATEGORIES, description='category')
save_category_widget = ipywidgets.Dropdown(options=SAVE_CATEGORIES, description='category')

In [ ]:
import numpy as np
from functools import partial

WIDTH = 80
HEIGHT = 80
SIZE = 8

check_image_button = ipywidgets.Button(description=f'{SIZE}個単位チェック')
check_next_images_button = ipywidgets.Button(description=f'[{SIZE}]>')
check_prev_images_button = ipywidgets.Button(description=f'<[-{SIZE}]')
check_start_index_widget = ipywidgets.IntText(description='開始位置')
check_end_index_widget = ipywidgets.IntText(description='終了位置')
check_image_count_widget = ipywidgets.IntText(description='最終画像位置')
check_update_button = ipywidgets.Button(description='更新')

# 画像を表示するウィジェット
snapshot_widgets = []
snapshot_button_widgets = []


def edit_image(index, b):
    global load_flag,no,check_flag
    no = check_no + index
    load_flag = True
    check_flag = False
    load_img(no)
    no_widget.value = no
    
for i in range(SIZE):
    image = ipywidgets.Image(width=WIDTH, height=HEIGHT)
    edit_button = ipywidgets.Button(description="編集", layout=ipywidgets.Layout(width=f'{WIDTH}px', height=f'30px'))
    edit_button.on_click(partial(edit_image, i))
    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
    image.value = bgr8_to_jpeg(black_image)
    snapshot_widgets.append(image)
    snapshot_button_widgets.append(VBox([edit_button,image]))

def get_load_dataset_length():
    global current_path
    xy_path = os.path.join(current_path, load_task_widget.value, load_datasets_widget.value, "xy")
    xy_filenames = get_file_names(xy_path)
    check_image_count_widget.value = len(xy_filenames)
    last_no = len(xy_filenames)
    return last_no

def next_images(c):
    global check_no,last_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_flag = True
    else:
        check_no += SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < last_no:
            load_images(check_no)
        else:
            check_next_images_button.disabled = False
            check_prev_images_button.disabled = False
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def prev_images(c):
    global check_no,check_flag
    check_next_images_button.disabled = True
    check_prev_images_button.disabled = True
    write_log(f"check_flag:"+str(check_flag))
    if check_flag == False:
        check_no = no_widget.value
        check_no -= SIZE
        check_flag = True
    else:
        check_no -= SIZE
        write_log(f"check_no: {check_no}")
    try:
        last_no = get_load_dataset_length()
        if check_no < 0:
            check_no = 0
        load_images(check_no)
    except:
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    
def load_images(c):
    global snapshot_widgets,check_no,last_no, current_path
    write_log("画像を" + str(SIZE) + "枚読込み、推論結果を付与します。")
    try:
        xy_path = os.path.join(current_path,load_task_widget.value,load_datasets_widget.value,"xy")
        xy_filenames = get_file_names(xy_path)
        last_no = len(xy_filenames)
        check_image_count_widget.value = last_no
        now_no = 0
        write_log(f"{xy_path}のデータセットを読み込みます。データ数(xy): {last_no}")
        for i in range(SIZE):
            now_no = check_no + i
            if now_no < last_no:
                try:
                    xy_name = xy_filenames[now_no]
                    img = cv2.imread(xy_name)
                    preprocessed = preprocess(img)
                    output = model(preprocessed).detach().cpu().numpy().flatten()
                    result_x = output[0]
                    result_y = output[1]
                    result_speed = output[3]
                    result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                    result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))
                    result_speed = int(IMG_HEIGHT * (result_speed / 2.0 + 0.5))
                    marked_img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)
                    marked_img = cv2.line(marked_img,(219,224-result_speed),(219,224),(0,140,255),3)
                    marked_img = cv2.putText(marked_img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))

                    snapshot_widgets[i].value = bgr8_to_jpeg(marked_img)
                    
                    time.sleep(10/1000)
                except Exception as e:
                    write_log(f"{e}")
                    black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                    snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
            else:
                black_image = np.zeros((HEIGHT, WIDTH, 3), dtype=np.uint8)
                snapshot_widgets[i].value = bgr8_to_jpeg(black_image)
        check_next_images_button.disabled = False
        check_prev_images_button.disabled = False
    except Exception as e:
        write_log(f"{e}")
    #get_jetson_nano_memory_usage()

def update_images(c):
    global check_no
    check_no = check_start_index_widget.value
    load_images(check_no)
    
check_no = 0
check_image_button.on_click(load_images)
check_prev_images_button.on_click(prev_images)
check_next_images_button.on_click(next_images)
check_update_button.on_click(update_images)

In [ ]:
movie_button = ipywidgets.Button(description='動画の作成')
movie_name_widget = ipywidgets.Text(description='動画名',value="run_video")

def make_movie(change):
    global model,current_path
    
    if not movie_name_widget.value.strip():
        write_log("ファイル名を指定してください。")
        return 
    write_log("動画を作成します。")
    path = os.path.join(current_path, "video/")
    if not os.path.exists(path):
        subprocess.call(['mkdir', '-p', path])
    output = path + movie_name_widget.value + ".mp4"
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    fps = int(30 / skip_movie_dropdown.value)
    outfh = cv2.VideoWriter(output, fourcc, fps, (224, 224))
    file_list = sorted(
        glob.glob(load_task_widget.value + '/' + load_datasets_widget.value + '/xy/*.jpg'),
        key=os.path.getmtime
    )
    
    xy_path = os.path.join(current_path, load_task_widget.value, load_datasets_widget.value, "xy")     
    file_list = os.listdir(xy_path)
    file_list = [os.path.join(xy_path, file_name) for file_name in file_list if file_name.endswith('.jpg')]    
    file_list = sorted(file_list, key=lambda f: extract_numbers(os.path.basename(f)))
    
    
    try:
        res_num = len(file_list)
        
        count = 0
        skip_movie = skip_movie_dropdown.value
        terminal_time = 1/(30/skip_movie)
        current_time = 0
        process_time = 0
        total_process_time = 0
        for i, file_name in enumerate(file_list):
            
            if i % skip_movie == 0:
                current_time += terminal_time
                img = cv2.imread(file_name)
                
                process_time = time.time()
                preprocessed = preprocess(img)
                output = model(preprocessed).detach().cpu().numpy().flatten()
                result_x = float(output[0])
                result_y = float(output[1])
                result_x = int(IMG_WIDTH * (result_x / 2.0 + 0.5))
                result_y = int(IMG_HEIGHT * (result_y / 2.0 + 0.5))    
                img = cv2.circle(img, (int(result_x), int(result_y)), 8, (255, 0, 0), 3)

                # Speed
                result_speed = output[3]
                result_speed = int(IMG_WIDTH * (result_speed / 2.0 + 0.5))
                if result_speed > 224:
                    result_speed = 244
                elif result_speed < 0:
                    result_speed = 0
                img = cv2.line(img,(218,0),(218,224),(0,0,0),5)
                img = cv2.line(img,(219,224-result_speed),(219,224),(0,140,255),3)
                img = cv2.putText(img,"speed:"+str(result_speed),(160,215),cv2.FONT_HERSHEY_SIMPLEX,0.3,(255,255,255))
                total_process_time += time.time() - process_time 
                
                if i % (skip_movie*10) == 0:
                    write_log(f"{current_time:.1f}秒まで完了, 推論処理平均: {total_process_time/10*1000:.1f}ms, {int(i/skip_movie)}枚目/{int(res_num/skip_movie)}枚中を処理中")
                    total_process_time = 0
                outfh.write(img)
                del img
    finally:
        # エラーが発生しても確実にリソースを解放する
        outfh.release()
        write_log("動画の出力が完了しました。")
        get_jetson_nano_memory_usage()

movie_button.on_click(make_movie)

In [ ]:
import subprocess
import re

used_memory_widget = ipywidgets.IntText(description='Useメモリ', value=1)
total_memory_widget = ipywidgets.IntText(description='全メモリ', value=1)
memory_button = ipywidgets.Button(description='使用メモリ量の取得')

def get_jetson_nano_memory_usage(event=None):
    command = 'tegrastats'
    try:
        process = subprocess.Popen(command, shell=True, stdout=subprocess.PIPE, stderr=subprocess.PIPE, universal_newlines=True)
        
        mem_usage_pattern = re.compile(r'RAM (\d+)/(\d+)MB')
        
        max_lines_to_read = 10
        for _ in range(max_lines_to_read):
            line = process.stdout.readline()
            if not line:
                break 
            matches = mem_usage_pattern.search(line)
            if matches:
                used_memory_widget.value = int(matches.group(1))
                total_memory_widget.value = int(matches.group(2))
                write_log("使用メモリ： " + str(used_memory_widget.value) + "/" + str(total_memory_widget.value))
                process.kill()
                return
        
        process.kill()  
        return

    except subprocess.CalledProcessError as e:
        return

get_jetson_nano_memory_usage()
memory_button.on_click(get_jetson_nano_memory_usage)

In [ ]:
separator = ipywidgets.HTML('<hr style="border-color:gray;margin:10px 0"/>')
title1 = ipywidgets.HTML('<b>【1.使用する推論モデル】</b> [New]は新規モデル。')
title2 = ipywidgets.HTML('<b>【2.読込元データセット】</b> アノテーションを実施するデータセットを選択。')
title3 = ipywidgets.HTML('<b>【3.保存先データセット】</b> データセットの保存先を選択。')
title4 = ipywidgets.HTML('<b>【4.アノテーションの実施】</b> 緑◯がアノテーション, 青◯がAIでの推論。車両の走らせたい場所で、画面をクリックすると保存先データセットのxyにデータが登録されます。Speedは[速度追加]で追加します。')
title5 = ipywidgets.HTML('<b>【5.学習】</b> EPOCH指定で学習できます。')
title6 = ipywidgets.HTML('<b>【6.評価動画の作成】</b> 動画を作成します。')

data_collection_widget = ipywidgets.VBox([
    separator,
    title1,
    ipywidgets.HBox([load_model_widget,load_model_time_widget,load_model_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title2,
    ipywidgets.HBox([load_datasets_widget,load_task_widget,load_image_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title3,
    ipywidgets.HBox([save_datasets_widget,save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([Label('datasetの新規作成'),datasets_name_widget,dataset_create_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title4,
    ipywidgets.HBox([no_widget,update_image_button,delete_image_button]), 
    ipywidgets.HBox([skip_dropdown,sleep_dropdown]),
    ipywidgets.HBox([speed_presets_box,picture_widget,ipywidgets.VBox([speed_slider,add_speed_button]),ipywidgets.VBox([add_xy0122_button,play_button,prev_image_button,Label(f'{SIZE}個単位での処理'),check_prev_images_button]),ipywidgets.VBox([add_xy224122_button,stop_button,next_image_button,ipywidgets.VBox([Label(f'')]),check_next_images_button])]),
    ipywidgets.HBox([save_datasets_widget,save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox(snapshot_button_widgets,),
    auto_advance_toggle,
    click_hold_ms_slider,
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title5,
    ipywidgets.HBox([save_datasets_widget,save_task_widget]),
    ipywidgets.HBox([datasets_xy_count_widget,datasets_speed_count_widget]),
    ipywidgets.HBox([epochs_widget,train_button,eval_button]),
    ipywidgets.HBox([progress_widget,loss_widget]),
    ipywidgets.HBox([save_model_name_widget,save_model_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
    separator,
    title6,
    ipywidgets.HBox([load_datasets_widget,load_task_widget]),
    ipywidgets.HBox([movie_name_widget,skip_movie_dropdown,movie_button]),
    ipywidgets.HBox([used_memory_widget,total_memory_widget,memory_button]),
    process_widget,
])
display(data_collection_widget)